# 4장 1강: 데이터 불균형과 평가지표 왜곡

## 실습 목표

이번 실습에서는 **Credit Card Fraud** 데이터를 이용하여 데이터 불균형이 분류 모델 평가에 어떤 영향을 주는지 확인합니다.

- `Class` 비율을 확인하여 불균형 정도를 파악합니다.
- Accuracy만 보면 생길 수 있는 착시를 확인합니다.
- Precision, Recall, F1-score, PR-AUC를 함께 확인합니다.
- `class_weight`, Random Undersampling, SMOTE를 적용하여 성능 변화를 비교합니다.
- threshold 변화에 따른 Precision / Recall 변화를 확인합니다.

> 이 실습에서는 4장 1강 교안에 나온 내용만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- imbalanced-learn
- 데이터: `creditcard.csv`

### 주요 컬럼

| 컬럼 | 의미 |
|---|---|
| `V1` ~ `V28` | 거래 정보를 변환한 숫자형 특성 |
| `Class` | 정상 거래 0 / 사기 거래 1 |

이번 실습에서는 모델 입력으로 `V1`부터 `V28`까지 사용하고, `Class`를 예측 대상으로 사용합니다.

## 실습 준비

1. 데이터 크기를 확인합니다.
2. `Class`의 개수와 비율을 확인합니다.
3. `V1`~`V28`을 X, `Class`를 y로 분리합니다.
4. Train / Test 데이터를 분리합니다.
5. 리샘플링은 **Train 데이터에만** 적용합니다.

In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

data = pd.read_csv("creditcard.csv")

print("데이터 크기:", data.shape)

print("\n[Class 개수]")
display(data["Class"].value_counts().to_frame("count"))

print("\n[Class 비율]")
display(data["Class"].value_counts(normalize=True).to_frame("ratio"))

features = [f"V{i}" for i in range(1, 29)]

X = data[features]
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain 크기:", X_train.shape)
print("Test 크기:", X_test.shape)

데이터 크기: (284807, 31)

[Class 개수]


,count
Class,
0,284315
1,492



[Class 비율]


,ratio
Class,
0,0.998273
1,0.001727



Train 크기: (227845, 28)
Test 크기: (56962, 28)


---

# 필수 1. Accuracy의 함정과 불균형 지표 확인

## 배경

Credit Card Fraud 데이터는 정상 거래가 매우 많고 사기 거래가 매우 적은 불균형 데이터입니다.

이런 상황에서는 Accuracy만 보면 모델이 매우 좋아 보일 수 있습니다.

## 문제 1. 모든 거래를 정상이라고 예측해보세요.

### 요구사항

1. Test 데이터의 모든 거래를 `0`으로 예측합니다.
2. Accuracy를 계산합니다.
3. Recall을 계산합니다.
4. Accuracy와 Recall 결과를 비교합니다.

### 확인 포인트

- 모든 거래를 정상이라고 예측해도 Accuracy가 매우 높게 나오는가?
- 사기 거래를 하나도 잡지 못하면 Recall은 어떻게 되는가?

In [10]:
# TODO
# 모든 거래를 정상(0)으로 예측하고 Accuracy / Recall을 계산하세요.

# 1. Test 데이터의 모든 거래를 정상(0)으로 예측
y_pred_all_normal = np.zeros(len(y_test), dtype=int)

# 2~3. Accuracy와 Recall 계산
acc_all_normal = accuracy_score(y_test, y_pred_all_normal)
recall_all_normal = recall_score(y_test, y_pred_all_normal)

print(f"Accuracy: {acc_all_normal:.4f}")
print(f"Recall:   {recall_all_normal:.4f}")

# 4. 비교를 위해 혼동행렬도 확인
print("\n[혼동행렬]")
print(confusion_matrix(y_test, y_pred_all_normal))

Accuracy: 0.9983
Recall:   0.0000

[혼동행렬]
[[56864     0]
 [   98     0]]




| 실제 정답 | 정상(0)으로 예측 | 사기(1)로 예측 | 내용 |
|---|---:|---:|---|
| 정상 0 | TN = 56,864 | FP = 0 | 정상 거래는 전부 맞춤 |
| 사기 1 | FN = 98 | TP = 0 | 사기 거래는 전부 틀림 |

-> TN : 정상을 정상으로 맞춘 건수, TP는 사기를 사기로 맞친 건수  
-> FP : 정상인데 사기라고 잘못 예측한 오탐  
-> FN : 사기인데 정상이라고 잘못 예측한 미탐  

### Q1. Accuracy가 높더라도 이 모델을 좋은 사기 탐지 모델이라고 보기 어려운 이유는 무엇인가요?
**답변:**  
데이터의 대부분이 정상거래이기 때문에 전부 다 정상거래로 예측해도 정확도(Accuracy)는 높음  
하지만 사기를 탐지해야하는데 Accuracy 하나만으로는 이 모델의 성능을 평가할 수 없음  
Recall, 정밀도 등도 같이 확인해야 모델의 성능을 평가할 수 있음  
-> 불균형 데이터에서는 Accuracy가 높아도 소수 클래스를 전형 탐지 못할 수도 있다.

## 문제 2. 기본 Logistic Regression의 여러 지표를 확인하세요.

### 요구사항

1. `LogisticRegression(max_iter=1000)` 모델을 학습합니다.
2. Test 데이터를 예측합니다.
3. Accuracy를 계산합니다.
4. Precision을 계산합니다.
5. Recall을 계산합니다.
6. F1-score를 계산합니다.
7. `predict_proba()`의 양성 클래스 확률로 PR-AUC를 계산합니다.
8. 혼동행렬을 확인합니다.

### 확인 포인트

- Accuracy 하나만 보는 것과 여러 지표를 함께 보는 것의 차이를 이해했는가?
- Precision과 Recall이 각각 어떤 오류와 연결되는지 설명할 수 있는가?
- PR-AUC가 불균형 데이터에서 왜 중요한지 이해했는가?

In [11]:
# TODO
# 기본 Logistic Regression의 Accuracy / Precision / Recall / F1 / PR-AUC / 혼동행렬을 확인하세요.

# 1. 기본 Logistic Regression 학습
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

# 2. Test 예측 (0/1 예측값과 사기 확률)
y_pred = lr.predict(X_test)
y_proba = lr.predict_proba(X_test)[:, 1]   # 양성(사기, 1) 클래스 확률

# 3~7. 지표 계산
lr_metrics = pd.DataFrame({
    "metric": ["Accuracy", "Precision", "Recall", "F1-score", "PR-AUC"],
    "value": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        average_precision_score(y_test, y_proba)
    ]
}).round(4)
display(lr_metrics)

# 8. 혼동행렬
print("[혼동행렬]")
print(confusion_matrix(y_test, y_pred))

,metric,value
0,Accuracy,0.9992
1,Precision,0.8289
2,Recall,0.6429
3,F1-score,0.7241
4,PR-AUC,0.7436


[혼동행렬]
[[56851    13]
 [   35    63]]


| 실제 정답 | 정상(0)으로 예측 | 사기(1)로 예측 | 내용 |
|---|---:|---:|---|
| 정상 0 | TN = 56,851 | FP = 13 | 정상 거래 56,864건 중 13건을 사기로 잘못 막음 |
| 사기 1 | FN = 35 | TP = 63 | 사기 거래 98건 중 63건을 잡고 35건을 놓침 |
|예측합계 | 56,886 | 76 | 56,962 |

TP : 사기 63건을 사기라고 올바르게 탐지한 경우  
FN : 사기 35건을 정상이라고 잘못 예측한 경우  
FP : 정상 13건을 사기라고 잘못 예측한 경우  
TN : 정상 56,851건을 정상이라고 올바르게 예측한 경우

사기라고 예측한 거래는 TP+FP=76건  
실제 사기 거래는 TP+FN=98건  

| 지표 | 계산식 | 결과 | 내용 |
|---|---|---:|---|
| Accuracy | (TP + TN) / 전체 = (63 + 56,851) / 56,962 | 0.9992 | 전체 거래 중 맞힌 비율. 정상 거래가 99.83%라 전부 정상 예측(0.9983)과 거의 차이 없음 |
| Precision | TP / (TP + FP) = 63 / (63 + 13) | 0.8289 | 사기로 예측한 76건 중 실제 사기 비율 82.8%  |
| Recall | TP / (TP + FN) = 63 / (63 + 35) | 0.6429 | 실제 사기 98건 중 잡아낸 비율 64.2% |
| F1-score | 2 × Precision × Recall / (Precision + Recall) | 0.7241 | Precision과 Recall의 조화평균. 둘 중 낮은 Recall 쪽으로 끌려 내려감 |
| PR-AUC | 정답과 사기 점수 | 0.7436 | 정상 거래(TN)에 영향받지 않고 사기를 얼마나 잘 구별하는지 평가. 무작위 기준값 약 0.0017보다 훨씬 높음 |

precision : 사기라고 말한 것을 얼마나 받을 수 있는지?  
Recall : 사기를 실제로 얼마나 놓치지않고 찾았는지?  
해석 : 모두 정상으로 예측한 기준선은 0건을 탐지해냄. 기본 모델같은 경우 63건을 찾고 35건을 놓쳤으며 정상 13건을 사기로 잘못 예측함  
사기를 사기로 탐지하는 능력은 있고 정확도도 매우 높은 편이지만  
Recall이 64%정도이기 때문에 사기를 거의 다 잡아낼 수 있는 모델이다. 라고 평가할 수는 없다.  

### Q2. 사기 거래를 놓치는 FN의 비용이 크다면 Precision과 Recall 중 무엇을 더 중요하게 봐야 하나요?

**답변:**  
Recall -> 실제 사기중에서 얼마나 사기를 정확하게 찾아냈는지를 나타내기 때문  
사기를 놓치는 FN을 중요하게 생각하신다면 Recall값을 최대한 높여서 만들어야함

---

# 필수 2. 불균형 처리 기법 비교

## 배경

교안에서는 불균형 처리 방법으로 다음 세 가지를 다룹니다.

- `class_weight`
- Random Undersampling
- SMOTE

이번 문제에서는 세 방법을 Train 데이터에만 적용한 뒤 같은 Test 데이터에서 성능을 비교합니다.

## 문제 1. class_weight를 적용하세요.

### 요구사항

1. `LogisticRegression(class_weight="balanced", max_iter=1000)`을 사용합니다.
2. Train 데이터로 학습합니다.
3. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 소수 클래스에 더 큰 가중치를 주면 Recall이 어떻게 변하는가?
- Precision과 Recall이 동시에 항상 좋아지는 것은 아니라는 점을 확인했는가?

In [12]:
# TODO
# class_weight="balanced" 모델을 학습하고 지표를 확인하세요.

# 같은 Test 데이터로 지표를 계산하는 함수 (뒤 문제에서도 재사용)
def evaluate_model(name, model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    y_proba = model.predict_proba(X_eval)[:, 1]
    return {
        "model": name,
        "precision": precision_score(y_eval, y_pred),
        "recall": recall_score(y_eval, y_pred),
        "f1": f1_score(y_eval, y_pred),
        "pr_auc": average_precision_score(y_eval, y_proba)
    }

# 1~2. class_weight="balanced" 모델 학습
lr_balanced = LogisticRegression(class_weight="balanced", max_iter=1000)
lr_balanced.fit(X_train, y_train)

# 3. Test 지표 계산 (문제 2의 기본 모델과 나란히 비교)
compare_rows = [
    evaluate_model("Logistic (기본)", lr, X_test, y_test),
    evaluate_model("class_weight=balanced", lr_balanced, X_test, y_test),
]
compare_df = pd.DataFrame(compare_rows).round(4)
display(compare_df)

print("[class_weight 혼동행렬]")
print(confusion_matrix(y_test, lr_balanced.predict(X_test)))

,model,precision,recall,f1,pr_auc
0,Logistic (기본),0.8289,0.6429,0.7241,0.7436
1,class_weight=balanced,0.0579,0.9184,0.1089,0.7092


[class_weight 혼동행렬]
[[55399  1465]
 [    8    90]]


balanced는 어떤 가중치를 부여하는지?

클래스 K의 학습 표본수가 NK 전체 학습 표본수가 N클래스 수가 K면
$$w_k \frac{n}{k n_k}$$
가중치가 0.50정도 사기 한 건이 약 289..14 정도의 수치  
같은 정도의 손실이라면 사기 한 건이 정상 한 건보다 약 577배 큰 비중을 가진다.  

Precision 처음에 82% -> 지금 5% -> 급격하게 사기라고 예측한 것중에 실제 사기였던 비율이 크게 낮아짐  
Recall 처음에 64% -> 지금 91% -> 실제 사기를 기존보다 더 잘 예측  
F1 처음에 72% -> 지금 10% -> Precision의 값이 매우 작아 종합값이 매우 떨어짐  

TP : 기존 모델 63건, 클래스 가중치 부여한 경우 90건 -> 사기 사건 27건 추가 탐지  
FN : 기존 모델 35건, 클래스 가중치 부여한 경우 8건 -> 사기라고 탐지 못한 경우 27건 감소  
FP : 기존 모델 13건, 클래스 가중치 부여한 경우 1,465건 -> 오탐 1,452건 증가  
TP + FP : 기존 모델 76건, 클래스 가중치 부여한 경우 1,555 -> 대부분 사기라고 예측을 해버림  

- 해석 : 모두 정상으로 예측한 기준선은 사기를 0건 찾았지만, 기본 모델은 63건을 찾고, 35건을 놓쳤고, 정상 13건을 오탐했다.  
사기 탐지 능력은 있으나 Accurcay가 95%를 넘는다고 매우 좋은 모델이라고 해석할 수는 없다.  
-> 실제 사기의 약 35%를 놓치기 때문이다.  

## 문제 2. Random Undersampling을 적용하세요.

### 요구사항

1. `RandomUnderSampler(random_state=42)`를 사용합니다.
2. **Train 데이터에만** `fit_resample()`을 적용합니다.
3. 리샘플링 전후 클래스 개수를 확인합니다.
4. Logistic Regression을 학습합니다.
5. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 다수 클래스 샘플이 줄어드는가?
- Test 데이터에는 Undersampling을 적용하지 않았는가?

In [ ]:
# TODO
# Random Undersampling을 Train 데이터에만 적용하세요.
# 1. 언더샘플러 생성
rus = RandomUnderSampler(random_state=42)

# 2. Train 데이터에만 리샘플링 적용 (Test는 건드리지 않음)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

# 3. 리샘플링 전후 클래스 개수 비교
count_compare = pd.DataFrame({
    "before": y_train.value_counts(),ㅠ
    "after": y_train_rus.value_counts()
})
display(count_compare)

# 4. 줄인 Train 데이터로 Logistic Regression 학습
lr_rus = LogisticRegression(max_iter=1000)
lr_rus.fit(X_train_rus, y_train_rus)

# 5. 원래 Test 데이터로 평가 (앞 결과와 함께 비교)
compare_rows.append(evaluate_model("Undersampling", lr_rus, X_test, y_test))
compare_df = pd.DataFrame(compare_rows).round(4)
display(compare_df)

print("[Undersampling 혼동행렬]")
print(confusion_matrix(y_test, lr_rus.predict(X_test)))


,before,after
Class,,
0,227451,394
1,394,394


,model,precision,recall,f1,pr_auc
0,Logistic (기본),0.8289,0.6429,0.7241,0.7436
1,class_weight=balanced,0.0579,0.9184,0.1089,0.7092
2,Undersampling,0.0376,0.9184,0.0722,0.6358


[Undersampling 혼동행렬]
[[54560  2304]
 [    8    90]]


- 왜 언더 샘플링 방법을 사용하는가?  
다수 클래스의 수를 줄이면 학습 자료에서 사기 표본의 상대점 비중이 높아짐  
처리할 자료의 양이 줄어서 학습 속도 향상을 기대할 수 있음  
다만, 현재 무작위로 남긴 394건이 정상 거래의 다양한 패턴을 충분히 반영하지 못하는 문제가 발생할 수 있음  

- 샘플링 비율, 남기는 표본, 모델 설정을 다르게 하면 모델의 결과가 달라질 수 있다.  
이번 언더 샘플링은 사기 탐지(Recall)의 값은 높았지만 오탐이 매우 많이 발생했다.  
언더 샘플링 자체가 잘못된 것이 아니라 데이터 셍이 언더 샘플링에 부적절했다고 판단한다.  

## 문제 3. SMOTE를 적용하세요.

### 요구사항

1. `SMOTE(random_state=42)`를 사용합니다.
2. **Train 데이터에만** `fit_resample()`을 적용합니다.
3. 리샘플링 전후 클래스 개수를 확인합니다.
4. Logistic Regression을 학습합니다.
5. Test 데이터에서 Precision, Recall, F1-score, PR-AUC를 계산합니다.

### 확인 포인트

- 소수 클래스 샘플이 증가하는가?
- SMOTE가 Test 데이터에 적용되지 않았는가?
- Recall 개선과 Precision 감소가 함께 나타날 수 있음을 확인했는가?

In [14]:
# TODO
# SMOTE를 Train 데이터에만 적용하세요.
# 1. SMOTE 생성
smote = SMOTE(random_state=42)

# 2. Train 데이터에만 리샘플링 적용 (Test는 건드리지 않음)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 3. 리샘플링 전후 클래스 개수 비교
count_compare_smote = pd.DataFrame({
    "before": y_train.value_counts(),
    "after": y_train_smote.value_counts()
})
display(count_compare_smote)

# 4. 늘린 Train 데이터로 Logistic Regression 학습
lr_smote = LogisticRegression(max_iter=1000)
lr_smote.fit(X_train_smote, y_train_smote)

# 5. 원래 Test 데이터로 평가 (앞 결과와 함께 비교)
compare_rows.append(evaluate_model("SMOTE", lr_smote, X_test, y_test))
compare_df = pd.DataFrame(compare_rows).round(4)
display(compare_df)

print("[SMOTE 혼동행렬]")
print(confusion_matrix(y_test, lr_smote.predict(X_test)))

,before,after
Class,,
0,227451,227451
1,394,227451


,model,precision,recall,f1,pr_auc
0,Logistic (기본),0.8289,0.6429,0.7241,0.7436
1,class_weight=balanced,0.0579,0.9184,0.1089,0.7092
2,Undersampling,0.0376,0.9184,0.0722,0.6358
3,SMOTE,0.0556,0.9184,0.1048,0.7188


[SMOTE 혼동행렬]
[[55334  1530]
 [    8    90]]




| 처리 | TP | FN | FP | 사기 예측 |
| --- | --- | --- | --- | --- |
| class weight | 90 | 8 | 1,465 | 1,555 |
| random undersampling | 90 | 8 | 2,304 | 2,394 |
| SMOTE | 90 | 8 | 1,530 | 1,620 |

-> class weight가 somte보다 오탐이 65건 더 적었고, AP는 smote가 약 0.718로 calss weight보다 약 0.09 높다.  

- smote의 한계점  
가까운 사기 표본 사이의 실제 사기 분포를 잘 나타낸다면 도움이 될 가능성이 높음  
그러나 이상치 주변이나 경계가 모호한 경우 의도치 않은 합성 표본이 발생할 수 있음  
합성 표본을 많이 만드는 것이 모든 성능평가 지표의 상승을 의미하지는 않음


### Q3. 리샘플링을 Test 데이터에도 적용하면 안 되는 이유는 무엇인가요?

**답변:**  

- test 데이터는 실제 배포 환경과 유사한 상황을 가정하고 실시간으로 들어오는 데이터들이 많다.  
그런데 인위적으로 리샘플링을 진행한다면 기존에 실제 환경과 다른 데이터셋들이 발생할 수 있다.  
과적합이 발생할 가능성이 있다.  
-> undersampling과 smote는 반드시 train 데이터에만 적용한다. 

---

# 과제 1. 불균형 처리 전후 성능과 threshold 비교

## 배경

필수 실습에서는 기본 모델과 세 가지 불균형 처리 방법을 확인했습니다.

이번 과제에서는 **기본 Logistic Regression과 class_weight 모델**을 비교하고, class_weight 모델에서 threshold에 따른 Precision / Recall 변화를 확인합니다.

> 과제는 필수 실습에서 사용한 지표와 방법을 그대로 비교하는 수준입니다.

## 요구사항

1. 기본 Logistic Regression과 `class_weight="balanced"` 모델의 다음 지표를 하나의 DataFrame으로 정리합니다.
   - Accuracy
   - Precision
   - Recall
   - F1-score
   - PR-AUC
2. 두 모델의 Recall과 Precision 차이를 설명합니다.
3. `class_weight="balanced"` 모델의 양성 확률을 사용합니다.
4. threshold를 `0.3`, `0.5`, `0.7`로 바꾸어 예측합니다.
5. 각 threshold에서 Precision과 Recall을 계산합니다.
6. threshold 변화에 따라 두 지표가 어떻게 달라지는지 설명합니다.
7. 사기 거래를 놓치는 FN의 비용이 크다고 가정할 때, threshold를 낮추는 방향이 어떤 의미인지 설명합니다.

8. 정상 거래만 예측해도 Accuracy가 높게 나오는 이유와 이 지표만 사용할 때의 문제를 설명합니다.
9. FN의 비용과 데이터 특성을 고려해 `class_weight`, Random Undersampling, SMOTE 중 우선 적용할 방법을 하나 선택하고 이유와 한계를 설명합니다. 추가 모델 구현은 요구하지 않습니다.
10. Q4~Q7에 모두 답합니다.

> 처리 기법 선택은 근거 있는 제안으로 작성합니다. Test 결과를 보고 처리 기법이나 threshold를 반복 조정하지 않으며, 실제 설정 선택에는 Train 내부 검증 또는 교차검증이 필요합니다.

### 확인 포인트

- 불균형 처리 전후를 Accuracy 하나가 아니라 여러 지표로 비교했는가?
- threshold를 낮추면 Recall이 높아지는 경향을 설명할 수 있는가?
- threshold를 높이면 Precision이 높아지는 경향을 설명할 수 있는가?
- 단일한 정답 threshold가 있는 것이 아니라 비용 구조에 따라 결정해야 한다는 점을 이해했는가?
- 클래스 비율 때문에 Accuracy가 사기 탐지 성능을 오해하게 할 수 있음을 설명했는가?
- FN 비용과 데이터 특성을 근거로 처리 기법의 선택 이유·한계를 제시했는가?


In [16]:
# TODO
# 기본 모델 / class_weight 모델 비교와 threshold 0.3, 0.5, 0.7 비교를 진행하세요.

# 두 모델 지표 비교
task_rows = []
for name, model in [("Logistic (기본)", lr), ("class_weight=balanced", lr_balanced)]:
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    task_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "pr_auc": average_precision_score(y_test, y_proba)
    })

task_compare_df = pd.DataFrame(task_rows).round(4)
display(task_compare_df)

# 3. class_weight 모델의 양성(사기) 확률
proba_balanced = lr_balanced.predict_proba(X_test)[:, 1]

threshold_rows = []
for th in [0.3, 0.5, 0.7]:
    # 4. 확률이 threshold 이상이면 사기(1)로 예측
    y_pred_th = (proba_balanced >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_th).ravel()

    # 5. Precision, Recall (혼동행렬 값도 함께)
    threshold_rows.append({
        "threshold": th,
        "precision": precision_score(y_test, y_pred_th),
        "recall": recall_score(y_test, y_pred_th),
        "TP": tp,
        "FP": fp,
        "FN": fn
    })

threshold_df = pd.DataFrame(threshold_rows).round(4)
display(threshold_df)



,model,accuracy,precision,recall,f1,pr_auc
0,Logistic (기본),0.9992,0.8289,0.6429,0.7241,0.7436
1,class_weight=balanced,0.9741,0.0579,0.9184,0.1089,0.7092


,threshold,precision,recall,TP,FP,FN
0,0.3,0.0259,0.9184,90,3390,8
1,0.5,0.0579,0.9184,90,1465,8
2,0.7,0.1146,0.8980,88,680,10


### Q4. 사기 거래를 놓치는 비용이 매우 크다면 threshold를 낮추는 방향은 어떤 장점과 단점이 있나요?

**답변:**  

- 장점: 사기 확률이 조금만 높아도 사기로 판정하여 더 많은 사기를 잡아 Recall이 높아지고 놓친 사기(FN)가 줄어듦  
-> (threshold 0.7 → 0.5에서 FN 10건 → 8건, Recall 0.8980 → 0.9184로 개선되어 FN 비용을 줄일 수 있음)

- 단점: 정상 거래도 사기로 판정되는 오탐(FP)이 늘어 Precision이 떨어짐  
-> (threshold 0.7 → 0.5에서 FP 680건 → 1,465건 / 0.5 → 0.3에서는 Recall 변화 없이 FP만 3,390건으로 늘어나 고객 불편과 확인 비용이 커질 수 있음)

- threshold는 FN과 FP의 비용을 비교해 정해야 하며, 실제 값은 Test가 아닌 Train 내부 검증이나 교차검증으로 선택해야함

### Q5. CTGAN / TVAE와 SMOTE의 가장 큰 차이를 한 문장으로 설명하세요.

**답변:**
- SMOTE: 비슷한 데이터 두 개를 골라 **그 사이에** 새 데이터를 만듦
- CTGAN / TVAE: 데이터 전체의 **패턴을 먼저 배운 뒤** 그 패턴대로 새 데이터를 만듦

### Q6. 정상 거래만 예측해도 Accuracy가 높게 나오는 이유와 이 지표만으로 사기 탐지 성능을 판단할 때의 문제를 설명하세요.

**답변:**  
- `정상 거래가 전체의 99.83%를 차지하기 때문에` 전부 정상 거래로 예측해도 정확도는 (Accuracy) 0.9983으로 높게 나옴

[이 지표만으로 사기 탐지 성능을 판단할 때의 문제]  
- 사기를 하나도 못 잡아도 정확도(Accuracy) 99.83%지만 재현율(Recall) 0.000%이기 때문에 실제 사기 98건은 잡지 못했다.  
- 모델 간 차이를 구분 못함: 전부 정상(0.9983)과 기본 모델(0.9992)의 차이는 0.0009뿐이지만 Recall은 0 vs 0.6429
- 판단이 거꾸로 될 수 있음: class_weight 모델은 사기를 27건 더 잡았지만 Accuracy는 0.9992 → 0.9741로 하락  
정확도(Accuracy)는 사기를 놓친 것(FN)과 정상을 잘못 막은 것(FP)을 똑같이 '틀린 1건'으로 세기때문에, 틀린 1건이 어떤 오류인지 알 수 없음  
-> 사기를 놓치는 비용이 정상을 잘못 막는 비용보다 큰 상황을 반영하지 못함


### Q7. 사기 거래를 놓치는 FN의 비용과 데이터 특성을 고려해 class_weight·Random Undersampling·SMOTE 중 우선 적용할 방법을 하나 선택하고, 선택 이유와 한계를 설명하세요.
**답변:**  
 
- 이유: 
  - 사기 거래를 놓치면 (FN), 사기 결제가 그대로 승인되어 금전적 손해가 발생할 수 있음  
  -> class_weight는 학습할 때 사기를 놓치는 경우에 더 큰 벌점을 주기 때문에 `class_weight`로 선택 (Recall 0.6429 → 0.9184, FN 35건 → 8건)

  - Random Undersampling : 정상 거래 227,451건 중 394건만 학습에 사용했으며, PR-AUC가 0.6358로 가장 낮았음.  
    class_weight는 데이터를 바꾸지 않아 정상 거래 227,451건의 데이터를 사용할 수 있음. 

  - SMOTE : 실제 사기 394건만으로 새 사기 데이터 약 22만 7천 건을 만들어 학습 데이터가 크게 늘었지만,  
성능은 class_weight와 비슷했으므로(Recall 0.9184로 동일, FP 1,530건 vs 1,465건) 데이터를 바꾸지 않는 class_weight를 우선 적용함

- 한계:  
  - Precision이 0.0579로 낮아 정상 거래를 사기로 막는 오탐(FP)이 1,465건 발생하므로 threshold를 함께 조정해야 함 (threshold 0.7에서 FP 680건, FN 10건)
  - PR-AUC는 기본 모델(0.7436)보다 낮은 0.7092로, 사기 구별 능력 자체를 높이는 방법은 아님
  - 가중치를 클래스 비율로만 정해 실제 FN·FP 비용 차이가 다를 수 있어, 가중치와 threshold는 Train 데이터의 교차검증으로 조정하는 것이 필요함


---

# 실습 마무리

아래 질문에 짧게 답해보세요.

1. 불균형 데이터에서 Accuracy 하나만 보면 위험한 이유는 무엇인가요?  
-> 다수 클래스만 잘 맞춰도 Accuracy가 높아질 수 있기 때문이다.  

2. Precision은 어떤 오류가 중요할 때 보는 지표인가요?  
-> 양성이라고 예측한 것중에 실제 양성 비율  

3. Recall은 어떤 오류가 중요할 때 보는 지표인가요?  
-> 실제 양성중에서 모델이 정확히 양성으로 판별한 비율  

4. class_weight, Undersampling, SMOTE는 각각 어떤 방식으로 불균형을 다루나요?  
-> Class_weight -> 소수 클래스 오류에 더 큰 가중치  
-> undersampling -> 다수 클래스 데이터 감소  
-> smote -> 소수 클래스 합성 데이터 증가

5. 리샘플링은 Train / Test 중 어디에만 적용해야 하나요?  
-> train 데이터셋에만 적용  

6. threshold를 낮추면 일반적으로 Recall과 Precision은 어떻게 변하나요?  
-> recall은 높아지고 precision은 낮아지는 경향이 있다.